# Benchmarks - Rust

The one Rust example from [docs/benchmarks.md](https://platob.github.io/yggdryl/benchmarks/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

### The pipeline those numbers measure

In [ ]:
use arrow_array::{Array, Int64Array};
use yggdryl::iceberg::Catalog;
use yggdryl::io::IOBase;
use yggdryl::local::Folder;
use yggdryl::text::TextLineOptions;
use yggdryl::Value;

let pattern = r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\S* \[(?<level>[^\]]+)\] \[(?<logger>[^\]]+)\] \[(?<thread_id>\d+)\] took=(?<latency_us>\d+)";
// The older extractor: same records, no thread column.
let archived = r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\S* \[(?<level>[^\]]+)\] \[(?<logger>[^\]]+)\] \[\d+\] took=(?<latency_us>\d+)";

let root = std::env::temp_dir().join(format!("yggdryl-docs-pipeline-{}", std::process::id()));
let _ = std::fs::remove_dir_all(&root);
let incoming = root.join("incoming");
let archive = root.join("archive");
std::fs::create_dir_all(&incoming)?;
std::fs::create_dir_all(&archive)?;

// Three rotated leaves in three codings; the second record spans a stack trace.
let first = concat!(
    "2024-02-01 10:00:00.000000 [ii] [engine] [3] took=120 fill 100 SYMB-0001\n",
    "2024-02-01 10:00:01.000000 [ee] [engine] [4] took=980 fill 101 SYMB-0002\n",
    "    at engine::match(order.rs:118)\n",
    "    at engine::step(order.rs:64)\n",
    "2024-02-01 10:00:02.000000 [ww] [router] [5] took=240 fill 102 SYMB-0003\n",
);
std::fs::write(incoming.join("app-0.log.gz"), yggdryl::gzip::dump(first.as_bytes())?)?;
std::fs::write(
    incoming.join("app-1.log"),
    b"2024-02-01 10:00:03.000000 [ee] [ledger] [6] took=770 fill 103 SYMB-0004\n",
)?;
std::fs::write(
    incoming.join("app-2.log.zst"),
    yggdryl::zstd::dump(b"2024-02-01 10:00:04.000000 [ii] [feed] [7] took=100 fill 104 SYMB-0005\n")?,
)?;
std::fs::write(
    archive.join("app-9.log.gz"),
    yggdryl::gzip::dump(b"2024-01-31 23:59:59.000000 [ee] [engine] [2] took=310 fill 099 SYMB-0000\n")?,
)?;

// 1. The extractor, and the table it implies - both before anything is read.
let options = TextLineOptions::with_pattern(pattern)?
    .try_with_custom_fields([("source", Value::from("gateway"))])?
    .with_byte_size(8 * 1024 * 1024);
let marked = options.schema().with_partition_fields(&["level"])?;

let catalog = Catalog::new(Folder::new(root.join("warehouse"))?);
let mut table = catalog.tables().create("logs.app", marked)?;

// 2, 3, 4. One handle per folder, and one lazy combine over the two.
let older = TextLineOptions::with_pattern(archived)?
    .try_with_custom_fields([("source", Value::from("archive"))])?;
let stream = yggdryl::arrow::combined(
    Folder::new(&incoming)?.into_arrow_lines(&options)?,
    Folder::new(&archive)?.into_arrow_lines(&older)?,
)?;

// 5. One commit, handed the reader itself - never a Vec of batches.
table.append(stream)?;

// The read-back asserts on the table, not on anything held in memory.
let mut rows = 0_usize;
let mut latency = 0_i64;
let mut threadless = 0_usize;
for batch in table.scan(None)? {
    let batch = batch?;
    rows += batch.num_rows();
    let took = batch
        .column_by_name("latency_us")
        .expect("the typed capture")
        .as_any()
        .downcast_ref::<Int64Array>()
        .expect("int64 by inference, not by declaration");
    latency += (0..batch.num_rows()).map(|row| took.value(row)).sum::<i64>();
    threadless += batch
        .column_by_name("thread_id")
        .expect("the merged column")
        .null_count();
}
// Five live records - not the seven lines they occupy - and one archived.
assert_eq!(rows, 6);
assert_eq!(latency, 120 + 980 + 240 + 770 + 100 + 310);
assert_eq!(threadless, 1, "the archived extractor had no thread column");

let reopened = catalog.table("logs.app")?;
assert_eq!(reopened.metadata().default_spec()?.fields[0].name, "level");
assert!(reopened.schema()?.get_field_by_name("source").is_some());

let _ = std::fs::remove_dir_all(&root);